# info score 경계 vs 실제 state 포화 검증 — gdn2-1.3B

**목표.** dynamic linear attention에서 **information score로 정한 state boundary**가, 실제로
recurrent state matrix가 **포화(saturation)하거나 전환(transition)하는 지점**과 얼마나 맞는지 검증한다.

**논리.** 같은 토큰 축 `t` 위에서 두 가지를 따로 재고 정렬시킨다.
1. **정보 점수 → 경계 `B_info`** — frozen 모델 1회 forward의 토큰별 신호(surprisal / epiplexity /
   entropy), 또는 **당신의 DSC 점수를 직접 주입**해 경계를 뽑음.
2. **상태 행렬 특성 `φ(S_t)`** — 각 위치의 층·헤드별 recurrent state `S_t`(dk×dv)에서
   eRank·‖·‖_F·σmax·stable-rank·write-magnitude·subspace-drift를 계산.

그리고 `B_info`가 `φ`의 변화점 `B_state`와 얼마나 겹치는지 **F1@tolerance + 무작위 null 대비 z-score +
Spearman**으로 정량화한다.

> **데이터는 당신이 넣습니다** — §1 셀에 텍스트/파일을 붙여넣으세요. 나머지는 그대로 실행.

---
### 주의 (실행 전 확인)
- **토크나이저**: config vocab=32000 → 기본 `TinyLlama/TinyLlama_v1.1`. 체크포인트 학습 토크나이저와
  다르면 surprisal이 틀어짐 → §0에서 `GDN2_TOKENIZER`로 교체.
- **fla 버전**: gdn2의 dscpkg chunk 커널이 fla 0.5.1과 비호환(`use_exp2`). `fused_recurrent`를 강제해
  회피하지만, 로드가 깨지면 dscpkg에 맞춘 fla env가 필요(§0 troubleshooting 참고).
- **per-position 상태**는 prefix 재실행(stride 단위)으로 뽑음 → `write_mag`은 stride 해상도.


## §0 설정 · 모델 로드

In [ ]:
import os, sys, math, json, time
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")          # fla-hub 401(xet) 회피
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# ── 디스크 우회 (중요) ────────────────────────────────────────────
# Triton은 CUDA util을 gcc로 컴파일하며 중간파일을 TMPDIR(기본 /tmp)에 씀.
# 루트 /tmp가 꽉 차면 "No space left on device"로 forward가 죽음 → 넉넉한 fs로 지정.
SCRATCH = os.environ.get("GDN2_SCRATCH", "/data2/sohyung/tmp")
os.makedirs(SCRATCH, exist_ok=True)
for _k in ("TMPDIR", "TEMP", "TMP"):
    os.environ[_k] = SCRATCH
os.environ.setdefault("TRITON_CACHE_DIR", os.path.join(SCRATCH, ".triton"))
os.environ.setdefault("HF_HOME", os.environ.get("HF_HOME", "/data2/sohyung/hf_cache"))
import numpy as np, torch

# ---- 경로/체크포인트 (환경에 맞게) ----
GDN2_HF        = "LLM-OS-Models2/gdn2-1.3b-paper-matched"
GDN2_CKPT_FILE = "model-95b.pth"                 # repo 파일명은 단수 model- (models- 아님)
CONFIG_NAME    = "gdn2_1.3B"                      # lit_gpt config 이름 (18L/18H/d2304/vocab32000)
TOKENIZER      = os.environ.get("GDN2_TOKENIZER", "TinyLlama/TinyLlama_v1.1")  # ← 학습 토크나이저 확인!
LIT_GPT_CANDS  = ["/root/dscpkg", "/home/sohyung/long-gdn/dsc"]  # dscpkg(lit_gpt) 위치 후보

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.bfloat16
RESULTS = os.path.join(os.getcwd(), "boundary_validation_results")
os.makedirs(RESULTS, exist_ok=True)
print("device:", DEVICE, "| dtype:", DTYPE, "| scratch:", SCRATCH)
assert DEVICE == "cuda", "gdn2 커널은 GPU 전용(Triton). GPU 노드에서 실행하세요."


In [ ]:
def _lit_gpt_path():
    p = os.environ.get("LIT_GPT_PATH")
    if p and os.path.isdir(os.path.join(p, "lit_gpt")):
        return p
    for c in LIT_GPT_CANDS:
        if os.path.isdir(os.path.join(c, "lit_gpt")):
            return c
    raise RuntimeError("lit_gpt(dscpkg) 못 찾음 — LIT_GPT_PATH 환경변수로 지정하세요.")


class Bundle:
    """gdn2 forward(logits) + recurrent-state 추출 래퍼."""
    def __init__(self, model, shared, n_layer):
        self.model, self.shared, self.n_layer = model, shared, n_layer

    @torch.no_grad()
    def logits(self, ids):
        return self.model(ids.to(DEVICE)).float()          # GPT는 logits를 바로 반환

    @torch.no_grad()
    def states(self, ids):
        from fla.models.utils import Cache
        self.shared["cache"] = Cache()
        self.model(ids.to(DEVICE))
        st = {}
        for i in range(self.n_layer):
            try:
                rs = self.shared["cache"][i]["recurrent_state"]
                elems = rs if isinstance(rs, (list, tuple)) else [rs]
                mats = [e.detach().float().cpu().reshape(-1, e.shape[-2], e.shape[-1])
                        for e in elems if torch.is_tensor(e) and e.dim() >= 2]
                if mats:
                    st[i] = torch.cat(mats, 0)              # (heads, dk, dv)
            except Exception:
                pass
        return st


def load_gdn2():
    sys.path.insert(0, _lit_gpt_path())
    from lit_gpt.config import Config
    from lit_gpt.model import GPT
    from lit_gpt.gdn2 import GatedDeltaNet2
    from huggingface_hub import hf_hub_download
    cfg = Config.from_name(CONFIG_NAME)
    m = GPT(cfg).to(DEVICE).to(DTYPE).eval()
    ck = torch.load(hf_hub_download(GDN2_HF, GDN2_CKPT_FILE), map_location="cpu", weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    missing, unexpected = m.load_state_dict(sd, strict=False)
    print(f"[load] missing={len(missing)} unexpected={len(unexpected)}  n_layer={cfg.n_layer}")

    # 상태 추출용 공유 cache 몽키패치 + fused_recurrent 강제
    SHARED = {"cache": None}
    _orig = GatedDeltaNet2.forward
    def patched(self, hidden_states, attention_mask=None, past_key_values=None, use_cache=False, **kw):
        pkv = SHARED["cache"] if SHARED["cache"] is not None else past_key_values
        uc  = True if SHARED["cache"] is not None else use_cache
        return _orig(self, hidden_states, attention_mask=attention_mask,
                     past_key_values=pkv, use_cache=uc, **kw)
    GatedDeltaNet2.forward = patched
    for i, mm in enumerate([x for x in m.modules() if isinstance(x, GatedDeltaNet2)]):
        mm.layer_idx = i
        mm.mode = "fused_recurrent"    # dscpkg chunk 커널의 use_exp2 비호환(fla0.5.1) 회피
    return Bundle(m, SHARED, cfg.n_layer)

bundle = load_gdn2()


In [ ]:
# 토크나이저
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(TOKENIZER)
print("tokenizer:", TOKENIZER, "| vocab:", tok.vocab_size)
# vocab이 32000과 크게 다르면 학습 토크나이저가 아님 → 위 GDN2_TOKENIZER 교체


In [ ]:
# --- 스모크 체크: 짧은 forward + 상태 추출이 도는지 ---
_smoke = tok("The capital of France is Paris. The capital of Japan is Tokyo.",
             return_tensors="pt").input_ids
with torch.no_grad():
    lg = bundle.logits(_smoke)
    nll = torch.nn.functional.cross_entropy(
        lg[0, :-1], _smoke[0, 1:].to(lg.device)).item()
st = bundle.states(_smoke)
print(f"logits {tuple(lg.shape)} | LM loss {nll:.2f} (자연 텍스트면 대략 <6)")
print(f"상태 추출 층수: {len(st)} | 예시 층0 shape: {tuple(next(iter(st.values())).shape)}")
# ── troubleshooting ──────────────────────────────────────────────
# * "chunk_gla_fwd_o_gk() unexpected 'use_exp2'" → chunk 커널 경로. fused_recurrent 강제가
#   안 먹은 것. mm.mode 확인. 그래도면 dscpkg에 맞춘 fla 버전 env 필요.
# * LM loss가 10+ 로 크면 토크나이저 불일치 가능성 → GDN2_TOKENIZER 교체.
# * "0 active drivers" → GPU 노드가 아님.


## §1 데이터 입력 — **여기에 당신 데이터를 넣으세요**

- `TEXT`에 직접 붙여넣거나, `DATA_FILE` 경로를 주면 파일에서 읽음.
- (선택) `GT_BOUNDARIES`에 사람이 아는 정답 경계(토큰 인덱스)를 주면 §4에서 함께 비교.
- (선택) `USER_BOUNDARIES`에 **당신 DSC 방법이 뽑은 경계**를 직접 주면 §2의 자동 검출 대신 그걸 사용.


In [ ]:
# ====================== 데이터 입력 ======================
TEXT = """여기에 분석할 텍스트를 붙여넣으세요. 서로 다른 주제/문서가 이어붙은
시퀀스일수록 (경계가 실제로 존재하므로) 검증이 선명합니다."""

DATA_FILE = None            # 예: "/data2/sohyung/mydata.txt" — 주면 TEXT 대신 파일 사용
MAX_TOKENS = 2048           # block_size(4096) 이하로. 길수록 prefix 재실행 비용↑

GT_BOUNDARIES   = None      # 예: [512, 1024, 1536]  (정답 경계 토큰 인덱스, 없으면 None)
USER_BOUNDARIES = None      # 예: [500, 1010, ...]   (당신 DSC 방법 경계, 주면 §2 자동검출 대체)
# =========================================================

if DATA_FILE:
    with open(DATA_FILE) as f:
        TEXT = f.read()

input_ids = tok(TEXT, return_tensors="pt").input_ids[:, :MAX_TOKENS]
T = input_ids.shape[1]
print(f"토큰 길이 T = {T}")
assert T >= 64, "너무 짧습니다. 최소 수백 토큰 권장."


## §2 정보 점수 → 경계 `B_info`

In [ ]:
@torch.no_grad()
def token_nll_bits(ids):
    """토큰별 surprisal (bits) — s_t = -log2 p(x_{t+1}|x_<=t). 길이 T-1."""
    logp = torch.log_softmax(bundle.logits(ids), -1)
    tgt = ids[:, 1:].to(logp.device)
    nll = -logp[:, :-1, :].gather(-1, tgt.unsqueeze(-1)).squeeze(-1)[0]
    return nll.cpu().numpy() / math.log(2)

@torch.no_grad()
def token_entropy_bits(ids):
    """토큰별 예측 엔트로피 (bits). 길이 T."""
    logp = torch.log_softmax(bundle.logits(ids), -1)
    H = -(logp.exp() * logp).sum(-1)[0].cpu().numpy() / math.log(2)
    return H

def epiplexity_excess(nll_bits, tail_frac=0.2):
    """점근선(뒤 tail_frac 평균) 위로 남는 초과 surprisal."""
    k = max(1, int(len(nll_bits) * tail_frac))
    asy = float(nll_bits[-k:].mean())
    return np.clip(nll_bits - asy, 0.0, None), asy

# ---- 정보 점수 선택 (원하는 것으로) ----
nll = token_nll_bits(input_ids)                 # (T-1,)
ent = token_entropy_bits(input_ids)[:-1]        # (T-1,) 로 맞춤
epi, _asy = epiplexity_excess(nll)

# ▼▼▼ 당신의 DSC 정보 점수를 여기에 붙여넣을 수 있음 (없으면 surprisal 사용) ▼▼▼
def custom_info_score(ids):
    return None   # 예: 당신 방법의 per-token score (길이 T-1) numpy 배열 반환
_custom = custom_info_score(input_ids)
# ▲▲▲

s_info = _custom if _custom is not None else nll   # 기본 = surprisal
INFO_NAME = "custom" if _custom is not None else "surprisal(NLL bits)"
print("info score:", INFO_NAME, "| len:", len(s_info))


In [ ]:
def detect_boundaries(score, k=None, min_gap=16, z_thresh=1.5):
    """score의 국소 최대(peak)를 경계로. k개(top-k) 또는 z>z_thresh, 최소간격 min_gap."""
    x = np.asarray(score, float)
    z = (x - x.mean()) / (x.std() + 1e-9)
    order = np.argsort(-z)                      # 높은 z부터
    picked = []
    for idx in order:
        if k is not None and len(picked) >= k:
            break
        if k is None and z[idx] < z_thresh:
            break
        if all(abs(idx - p) >= min_gap for p in picked):
            picked.append(int(idx))
    return sorted(picked)

if USER_BOUNDARIES is not None:
    B_info = sorted(int(b) for b in USER_BOUNDARIES)
    print("USER_BOUNDARIES 사용:", B_info)
else:
    B_info = detect_boundaries(s_info, k=None, min_gap=max(16, T // 40), z_thresh=1.5)
    print(f"자동 검출 경계 {len(B_info)}개:", B_info)


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(s_info, lw=0.8, color="steelblue")
for b in B_info:
    ax.axvline(b, color="crimson", lw=1, alpha=.7)
if GT_BOUNDARIES:
    for b in GT_BOUNDARIES:
        ax.axvline(b, color="green", ls="--", lw=1.2, alpha=.7)
ax.set_title(f"info score = {INFO_NAME}  (red=B_info, green=GT)")
ax.set_xlabel("token position t"); ax.set_ylabel("score")
fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "info_score_boundaries.png"), dpi=120)
print("saved info_score_boundaries.png"); plt.close(fig)


## §3 상태 행렬 특성 궤적 `φ(S_t)`

In [ ]:
def state_metrics(st):
    """st={layer: (heads,dk,dv)} → 층·헤드 평균 특성 + 층별 eRank + flatten 벡터."""
    er, fr, sm, sr = [], [], [], []
    per_layer_er, flat_parts = {}, []
    for li, S in sorted(st.items()):
        M = S.reshape(-1, S.shape[-2], S.shape[-1]).float()
        le = []
        for m in M:
            s = torch.linalg.svdvals(m); s = s[s > 0]
            if s.numel() == 0:
                le.append(0.0); continue
            p = s / s.sum()
            le.append(float(torch.exp(-(p * torch.log(p)).sum())))   # eRank
            fr.append(float(m.norm()))
            sm.append(float(s[0]))
            sr.append(float((m.norm() ** 2) / (s[0] ** 2 + 1e-9)))   # stable rank
        per_layer_er[li] = float(np.mean(le)); er.extend(le)
        flat_parts.append(M.reshape(-1))
    return {"erank": float(np.mean(er)), "fro": float(np.mean(fr)),
            "smax": float(np.mean(sm)), "stable_rank": float(np.mean(sr)),
            "per_layer_erank": per_layer_er, "flat": torch.cat(flat_parts)}


def state_trajectory(ids, stride=32, min_t=8):
    """prefix[:t]를 grid로 훑으며 상태 특성. write_mag/drift는 직전 grid 대비."""
    T = ids.shape[1]
    grid = list(range(min_t, T + 1, stride))
    if grid[-1] != T:
        grid.append(T)
    traj, prev = [], None
    for j, t in enumerate(grid):
        m = state_metrics(bundle.states(ids[:, :t]))
        flat = m.pop("flat")
        if prev is None:
            wm = dr = float("nan")
        else:
            wm = float((flat - prev).norm() / (flat.norm() + 1e-9))
            dr = 1.0 - float(torch.nn.functional.cosine_similarity(
                flat.unsqueeze(0), prev.unsqueeze(0)).item())
        prev = flat
        m.update({"t": t, "write_mag": wm, "drift": dr})
        traj.append(m)
        if j % 10 == 0:
            print(f"  t={t}/{T}  eRank={m['erank']:.2f}")
    return grid, traj

STRIDE = max(16, T // 64)          # 약 64 grid point
grid, traj = state_trajectory(input_ids, stride=STRIDE)
print(f"grid points: {len(grid)} (stride={STRIDE})")


In [ ]:
ts   = np.array([m["t"] for m in traj])
curves = {k: np.array([m[k] for m in traj], float)
          for k in ["erank", "fro", "smax", "stable_rank", "write_mag", "drift"]}

fig, axes = plt.subplots(3, 2, figsize=(13, 8), sharex=True)
titles = {"erank": "eRank (포화)", "fro": "‖S‖_F (에너지)", "smax": "σmax",
          "stable_rank": "stable rank", "write_mag": "write magnitude ‖ΔS‖/‖S‖",
          "drift": "subspace drift (1-cos)"}
for ax, (k, ttl) in zip(axes.ravel(), titles.items()):
    ax.plot(ts, curves[k], "-o", ms=3, color="darkorange")
    for b in B_info:
        ax.axvline(b, color="crimson", lw=1, alpha=.6)
    if GT_BOUNDARIES:
        for b in GT_BOUNDARIES:
            ax.axvline(b, color="green", ls="--", lw=1, alpha=.6)
    ax.set_title(ttl, fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("token position t")
fig.suptitle("state matrix 특성 vs 위치 (red=B_info, green=GT)", y=1.0)
fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "state_characteristics.png"), dpi=120)
print("saved state_characteristics.png"); plt.close(fig)


In [ ]:
# 층별 eRank 히트맵 — 어느 층이 언제 포화하나
layers = sorted({li for m in traj for li in m["per_layer_erank"]})
H = np.array([[m["per_layer_erank"].get(li, np.nan) for m in traj] for li in layers])
fig, ax = plt.subplots(figsize=(13, 4))
im = ax.imshow(H, aspect="auto", origin="lower",
               extent=[ts[0], ts[-1], layers[0], layers[-1]], cmap="viridis")
for b in B_info:
    ax.axvline(b, color="red", lw=1, alpha=.7)
fig.colorbar(im, ax=ax, label="eRank")
ax.set_xlabel("token position t"); ax.set_ylabel("layer")
ax.set_title("층별 eRank (red=B_info)")
fig.tight_layout(); fig.savefig(os.path.join(RESULTS, "layer_erank_heatmap.png"), dpi=120)
print("saved layer_erank_heatmap.png"); plt.close(fig)


## §4 정렬 분석 — `B_info`가 상태 변화점과 맞는가?

In [ ]:
def state_changepoints(ts, curve, k=None, min_gap_pts=1, z_thresh=1.5):
    """상태 곡선의 |1차차분| peak → 변화점(토큰 위치). eRank 꺾임/write-mag 급변 포착."""
    d = np.abs(np.gradient(np.nan_to_num(curve)))
    z = (d - d.mean()) / (d.std() + 1e-9)
    order = np.argsort(-z); picked = []
    for i in order:
        if k is not None and len(picked) >= k:
            break
        if k is None and z[i] < z_thresh:
            break
        if all(abs(i - p) >= min_gap_pts for p in picked):
            picked.append(i)
    return sorted(int(ts[i]) for i in picked)


def alignment(B_info, B_state, T, w, n_shuffle=1000, rng=None):
    """B_info vs B_state: tolerance w 이내 매칭 F1 + 무작위 경계 null 대비 z."""
    rng = rng or np.random.default_rng(0)
    def f1(A, Bs):
        if not A or not Bs:
            return 0.0
        tp = sum(any(abs(a - b) <= w for b in Bs) for a in A)
        prec = tp / len(A); rec = sum(any(abs(a - b) <= w for a in A) for b in Bs) / len(Bs)
        return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)
    obs = f1(B_info, B_state)
    null = np.array([f1(sorted(rng.integers(0, T, len(B_info)).tolist()), B_state)
                     for _ in range(n_shuffle)])
    z = (obs - null.mean()) / (null.std() + 1e-9)
    return {"f1": obs, "null_f1_mean": float(null.mean()),
            "null_f1_std": float(null.std()), "z": float(z),
            "p_ge": float((null >= obs).mean())}

from scipy.stats import spearmanr

W = max(STRIDE, T // 40)         # 매칭 허용 오차 (토큰)
report = {"info_name": INFO_NAME, "T": int(T), "tolerance_w": int(W),
          "B_info": B_info, "n_grid": len(grid)}

# 각 상태 특성의 변화점과 B_info 정렬
for k in ["erank", "write_mag", "drift", "fro", "smax", "stable_rank"]:
    B_state = state_changepoints(ts, curves[k], z_thresh=1.5)
    al = alignment(B_info, B_state, T, W)
    report[f"align_{k}"] = {"B_state": B_state, **al}
    print(f"[{k:11s}] F1={al['f1']:.2f}  null={al['null_f1_mean']:.2f}  z={al['z']:+.2f}  p={al['p_ge']:.3f}")

# info score(=grid에서 재샘플) vs write_mag 상관
s_info_grid = np.interp(ts, np.arange(len(s_info)), s_info)
for k in ["write_mag", "drift", "erank"]:
    m = ~np.isnan(curves[k])
    rho, p = spearmanr(s_info_grid[m], curves[k][m])
    report[f"spearman_info_vs_{k}"] = {"rho": float(rho), "p": float(p)}
    print(f"Spearman(info, {k}) = {rho:+.2f} (p={p:.3f})")

if GT_BOUNDARIES:
    report["align_info_vs_GT"]  = alignment(B_info, sorted(GT_BOUNDARIES), T, W)
    print("info vs GT:", report["align_info_vs_GT"])


In [ ]:
with open(os.path.join(RESULTS, "alignment_report.json"), "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print("saved alignment_report.json")
print(json.dumps({k: v for k, v in report.items() if k.startswith("align_") or k.startswith("spearman")},
                 indent=2, ensure_ascii=False))


## §5 해석 (판정 틀)

각 특성 `k`에 대해:
- **z ≫ 0 이고 F1 ≫ null_f1** → info score 경계가 그 상태 특성의 변화점과 우연 이상으로 정렬됨.
  → info score가 **실제 state 전환/포화를 잡고 있다**.
- **z ≈ 0** → 우연 수준. info score는 그 특성과 무관 (경계 기준으로 부적절).
- **Spearman(info, write_mag) > 0** → surprisal이 높은 곳에서 상태가 많이 갱신됨(=쓰기 활발) →
  경계 근거로 타당. 음/무상관이면 정보량↑가 상태 변화로 이어지지 않음.

**주의/한계**
- eRank는 앞선 실험들에서 recall/용량과 decouple됨(REPORT 참고) — 여기선 "포화의 *한* 신호"로만 보고,
  write_mag/drift(상태가 실제로 바뀌는 정도)를 더 신뢰. 여러 특성의 합의를 보라.
- prefix 재실행이라 `write_mag`은 stride 해상도. 토큰 단위가 필요하면 incremental 추출로 교체.
- 서로 다른 문서를 이어붙인 데이터(경계가 물리적으로 존재)로 하면 `GT_BOUNDARIES`와 직접 대조 가능.
